# Flex

This notebook covers the "Flex" section of Chapter 7, "Custom Modules, Types, and Adapters" (under "Built-In Modules", pages 170–171): `dspy.Flex` makes a module's Python implementation something GEPA can optimize, so you do not have to choose an inference strategy before you run the optimizer.

**Environment variables**: `OPENAI_API_KEY` (loaded from `.env`).

**Models**: `openai/gpt-5.6-luna`, the book's default. The opt-in GEPA compile also uses `openai/gpt-5.6-sol` as the reflection model.

**Cost**: one short model call by default. The GEPA compile from page 170 is opt-in: with `auto="light"` and this notebook's five validation examples, GEPA budgets about 400 program runs. Each run starts a sandbox, so the compile takes a few minutes, and every rewrite of the implementation is a call to the stronger model. DSPy caches repeated model calls, so most of the 400 runs do not reach the API.

**Install**: the next cell installs the repository's pinned packages. If you set up the environment with `uv sync --frozen` as described in the Preface, everything is already installed and you can skip it; running it anyway is harmless. With pip or Google Colab, run the notebook from its `chapter07/` folder inside the cloned repository so that `../requirements.txt` resolves.

## Deno is required

Flex runs its implementation in a Deno + Pyodide sandbox, both before and after optimization. Install Deno before running this notebook (in a terminal, not as a notebook cell):

```
curl -fsSL https://deno.land/install.sh | sh
```

Flex was introduced in DSPy 3.3.0 and is experimental; this repository pins `dspy==3.3.0`.

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM("openai/gpt-5.6-luna"))

## Before compilation

Without tools, a Flex program starts as a single `Predict`. Building the program and printing its starting implementation makes no model call.

In [ ]:
baseline = dspy.Flex("question: str -> answer: str")
print(baseline.module_src)

Until it is compiled, the program answers the way a plain `Predict` would. The call below makes one model call and runs the implementation above in the sandbox, so it is also a quick check that Deno is installed.

In [ ]:
result = baseline(question="What is 15% of 240?")
print(result.answer)

## The pieces GEPA needs

> **Book vs. notebook:** The listing on page 170 reuses `metric`, `reflection_lm`, `trainset`, and `valset` from the GEPA examples in Chapter 6. The cell below defines small stand-ins with the same names so this notebook runs on its own. The stand-in `metric` returns feedback on every call, which gives GEPA text to reflect on when it rewrites a Flex implementation.

The stand-in data mixes short factual questions with arithmetic word problems, and the metric accepts only the exact short answer: `210`, not `210 minutes`. A single `Predict` tends to answer the word problems with a unit or a full sentence, so the starting program misses several examples and GEPA has failures to reflect on. `reflection_lm` is `openai/gpt-5.6-sol`, the stronger of the book's two default models.

In [ ]:
qa_pairs = [
    ("What is the capital of France?", "Paris"),
    ("How many minutes are in 3.5 hours?", "210"),
    ("What is 15% of 240?", "36"),
    ("A train travels at 60 km per hour for 2.5 hours. How many km does it cover?", "150"),
    ("A shirt costs $40 and is discounted by 25%. What is the sale price in dollars?", "30"),
    ("Which planet is known as the Red Planet?", "Mars"),
    ("How many seconds are in 2.5 minutes?", "150"),
    ("What is 1,250 minus 375?", "875"),
    ("A car uses 8 liters of fuel per 100 km. How many liters does it use for 250 km?", "20"),
    ("A rectangle is 12 cm wide and 7 cm tall. What is its area in square cm?", "84"),
]
examples = [
    dspy.Example(question=question, answer=answer).with_inputs("question")
    for question, answer in qa_pairs
]
trainset, valset = examples[:5], examples[5:]


def normalize(text):
    return str(text).strip().rstrip(".").lower()


def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    actual = getattr(prediction, "answer", "")
    score = 1.0 if normalize(actual) == normalize(example.answer) else 0.0
    feedback = (
        "Correct."
        if score
        else f"Expected exactly '{example.answer}' with no extra words, got '{actual}'."
    )
    return dspy.Prediction(score=score, feedback=feedback)


reflection_lm = dspy.LM("openai/gpt-5.6-sol", temperature=1.0)

## Optimizing the implementation with GEPA

The next cell is the listing from page 170. Given a Flex program, GEPA's reflection model can rewrite the whole implementation rather than only its instructions, for example by adding predictors, helper functions, or routing between inputs.

> **Book vs. notebook:** The listing on page 170 runs the compile directly. Here the same lines sit under an `if RUN_FLEX_OPTIMIZATION:` flag, because the compile runs the program about 400 times in the sandbox, which takes a few minutes, and calls the stronger reflection model. Set `RUN_FLEX_OPTIMIZATION = True` to run it.

In [ ]:
RUN_FLEX_OPTIMIZATION = False  # set to True to run the GEPA compile below

if RUN_FLEX_OPTIMIZATION:
    program = dspy.Flex(
        "question: str -> answer: str"
    )

    optimized = dspy.GEPA(
        metric=metric,
        auto="light",
        reflection_lm=reflection_lm
    ).compile(
        program,
        trainset=trainset,
        valset=valset
    )

    print(optimized.module_src)
else:
    print("Skipped. Set RUN_FLEX_OPTIMIZATION = True to run the GEPA compile.")

## Reading the result

While the compile runs, GEPA logs every iteration. The `Average Metric` progress bars count program runs that completed, so they read 100% even when an answer is wrong. The scores from `metric` are in the `Iteration N:` log lines, starting with `Base program full valset score`. A line that says `All subsample scores perfect. Skipping.` means the selected candidate already answers that minibatch correctly, so there is nothing to reflect on.

After compilation, `optimized.module_src` holds the implementation GEPA selected. Compare it with the starting implementation printed in "Before compilation". On a dataset this small GEPA may change only the instructions, add a few lines of Python, or keep the starting implementation if no candidate scores higher. Either way the result is ordinary DSPy module code that you can read, diff, and save along with the rest of the program.

Model-written code does not run in your own Python process: Flex executes it inside the sandboxed interpreter, and `max_predictor_calls` (100 by default) limits how many predictor calls one run of the program can make. Lower it for a tighter guard against runaway programs, for example `dspy.Flex("question: str -> answer: str", max_predictor_calls=20)`.

Chapter 7 recommends starting with `Predict` and reaching for Flex once your evaluation results show that the module's structure, rather than its instructions, is what limits the program.